<a href="https://colab.research.google.com/github/sam-wahid/vlm-llm-segmentation/blob/main/CLIPbenchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q torch torchvision
!pip install -q open_clip_torch
!pip install -q scikit-learn tqdm matplotlib

In [ ]:
import time
import torch
import open_clip
import numpy as np
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.Grayscale(num_output_channels=3),

    transforms.ToTensor()
])

dataset = datasets.FashionMNIST(

    root="./data",
    train=False,
    download=True,
    transform=transform
)

loader = DataLoader(

    dataset,
    batch_size=32,
    shuffle=False
)

print("Total Images:", len(dataset))

print("Classes:", dataset.classes)

In [ ]:
clip_model, _, preprocess = open_clip.create_model_and_transforms(

    'ViT-B-32',
    pretrained='openai'
)

clip_model = clip_model.to(device)

clip_model.eval()

print("CLIP MODEL LOADED")

In [ ]:
def benchmark_clip_model(model, loader):

    embeddings = []
    all_labels = []

    total_time = 0
    total_images = 0

    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()

    with torch.no_grad():

        for images, labels in tqdm(loader):

            images = images.to(device)

            start = time.time()

            features = model.encode_image(images)

            end = time.time()

            total_time += (end - start)

            total_images += images.size(0)

            features = features.cpu().numpy()

            embeddings.append(features)

            all_labels.append(labels.numpy())

    embeddings = np.concatenate(embeddings, axis=0)

    all_labels = np.concatenate(all_labels, axis=0)

    fps = total_images / total_time

    latency = total_time / total_images

    if torch.cuda.is_available():

        gpu_memory = (
            torch.cuda.max_memory_allocated() / 1024**3
        )

    else:
        gpu_memory = 0

    return {

        "embeddings": embeddings,
        "labels": all_labels,
        "fps": fps,
        "latency": latency,
        "gpu_memory": gpu_memory
    }

In [ ]:
clip_results = benchmark_clip_model(

    clip_model,
    loader
)

In [ ]:
X = clip_results["embeddings"]

y = clip_results["labels"]

split = int(0.8 * len(X))

X_train = X[:split]
X_test = X[split:]

y_train = y[:split]
y_test = y[split:]

knn = KNeighborsClassifier(n_neighbors=5)

knn.fit(X_train, y_train)

predictions = knn.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

In [ ]:
print("\n========== CLIP FINAL RESULTS ==========")

print("Accuracy      :", round(accuracy * 100, 2), "%")

print("FPS           :", round(clip_results["fps"], 2))

print("Latency       :", round(clip_results["latency"], 5))

print("GPU Memory GB :", round(clip_results["gpu_memory"], 2))

In [ ]:
metrics = [

    round(accuracy * 100, 2),

    round(clip_results["fps"], 2),

    round(clip_results["latency"], 5),

    round(clip_results["gpu_memory"], 2)
]

metric_names = [

    "Accuracy",
    "FPS",
    "Latency",
    "GPU Memory"
]

plt.figure(figsize=(8,5))

plt.bar(metric_names, metrics)

plt.title("CLIP Benchmark Results")

plt.show()